# Comparing Machine Learning Classification Methods

### Goal
Use **the same dataset and the same train/test split** to compare how different machine learning classifiers perform.

We will compare:

- Logistic Regression
- k-Nearest Neighbors (KNN)
- Support Vector Machine (SVM)
- Decision Tree
- Random Forest
- Gaussian Naive Bayes
- Multilayer Perceptron (MLP)

Dataset: **Breast Cancer Wisconsin** from `scikit-learn`.

The target is binary:

- `0` – malignant
- `1` – benign

> The important idea: **no single machine learning method is always the best**.  
> Performance depends on the dataset, preprocessing, hyperparameters, and evaluation metric.


In [ ]:
# If needed, uncomment this line:
# !pip install numpy pandas matplotlib scikit-learn

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

# In machine learning, the random state is a seed value that is used to initialize the random number generator. 
# This ensures that the same random numbers are generated every time the code is run, which can be helpful for debugging and reproducibility.
# Setting random_state=42 is a popular coding tradition.
RANDOM_STATE = 42


## 1. Load the dataset

The dataset contains numerical measurements computed from digitized images of breast masses.

For this exercise, we treat it simply as a **binary classification dataset**.


In [ ]:
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print("X shape:", X.shape)
print("Classes:", dict(enumerate(data.target_names)))
print()
print("Class counts:")
print(y.value_counts().sort_index())

X.head()


## 2. Use exactly the same train/test split for every classifier

This is essential for a fair comparison.

`stratify=y` keeps approximately the same class proportions in the training and test sets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))


## 3. Define the models

Some algorithms are sensitive to feature scale, so they are placed inside a pipeline with `StandardScaler`.

Tree-based methods do not require scaling.


In [ ]:
# Define all classifiers in one dictionary.
# Pipelines are used when a model benefits from feature standardization.
#
# StandardScaler():
#   transforms each feature approximately to mean = 0 and standard deviation = 1.
#   This is especially important for distance- or scale-sensitive methods
#   such as Logistic Regression, KNN, SVM, and MLP.

models = {

    # ------------------------------------------------------------
    # 1. LOGISTIC REGRESSION
    # ------------------------------------------------------------
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=2000,              # Maximum number of optimization iterations.
                                        # A larger value gives the algorithm more time
                                        # to converge if the default is not sufficient.

            random_state=RANDOM_STATE   # Fixes the random seed for reproducibility.
                                        # With the same seed, stochastic parts of the
                                        # algorithm behave consistently between runs.
        ))
    ]),


    # ------------------------------------------------------------
    # 2. k-NEAREST NEIGHBORS (KNN)
    # ------------------------------------------------------------
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(
            n_neighbors=5               # Number of nearest training samples ("k")
                                        # used to classify a new observation.
                                        # Small k -> more flexible, more sensitive to noise.
                                        # Large k -> smoother decision boundary.
        ))
    ]),


    # ------------------------------------------------------------
    # 3. SUPPORT VECTOR MACHINE (SVM)
    # ------------------------------------------------------------
    "SVM (RBF)": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(
            kernel="rbf",               # Kernel function.
                                        # "rbf" (Radial Basis Function) allows the SVM
                                        # to create nonlinear decision boundaries.

            probability=True,           # Enables predict_proba(), so class probabilities
                                        # can be estimated and ROC AUC can be calculated
                                        # using probability scores.

            random_state=RANDOM_STATE   # Fixes randomness used internally when
                                        # probability estimates are enabled.
        ))
    ]),


    # ------------------------------------------------------------
    # 4. DECISION TREE
    # ------------------------------------------------------------
    "Decision Tree": DecisionTreeClassifier(
        random_state=RANDOM_STATE       # Makes random choices made while building
                                        # the tree reproducible.
    ),


    # ------------------------------------------------------------
    # 5. RANDOM FOREST
    # ------------------------------------------------------------
    "Random Forest": RandomForestClassifier(
        n_estimators=300,               # Number of Decision Trees in the forest.
                                        # More trees usually make predictions more stable,
                                        # but increase computation time.

        random_state=RANDOM_STATE       # Makes the random construction of the forest
                                        # reproducible between runs.
    ),


    # ------------------------------------------------------------
    # 6. GAUSSIAN NAIVE BAYES
    # ------------------------------------------------------------
    "Gaussian Naive Bayes": GaussianNB(),
    # No parameters are explicitly specified here.
    # Therefore, scikit-learn uses the default GaussianNB settings.
    # The method assumes that each feature follows a Gaussian (normal)
    # distribution within each class.


    # ------------------------------------------------------------
    # 7. MULTILAYER PERCEPTRON (MLP)
    # ------------------------------------------------------------
    "MLP Neural Network": Pipeline([
        ("scaler", StandardScaler()),
        ("model", MLPClassifier(
            hidden_layer_sizes=(50,),   # Architecture of the hidden layer(s).
                                        # (50,) means one hidden layer with 50 neurons.
                                        # For example, (50, 20) would mean two hidden
                                        # layers containing 50 and 20 neurons.

            max_iter=2000,              # Maximum number of training iterations (epochs
                                        # over the optimization procedure). A larger value
                                        # gives the optimizer more opportunities to converge.

            random_state=RANDOM_STATE   # Fixes the random initialization and other
                                        # stochastic operations for reproducibility.
        ))
    ])
}

print("Number of classifiers:", len(models))


## 4. Train and evaluate all classifiers

We calculate several metrics because **accuracy alone does not tell the whole story**.

- **Accuracy** – proportion of all correctly classified samples
- **Balanced accuracy** – average recall across classes
- **Precision** – how many predicted positives were correct
- **Recall** – how many actual positives were detected
- **F1-score** – balance between precision and recall
- **ROC AUC** – ranking/separation ability across classification thresholds


In [ ]:
results = []
trained_models = {}

for name, model in models.items():
    start = time.perf_counter()

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    elapsed = time.perf_counter() - start

    # Probability / decision score for ROC AUC
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        y_score = model.decision_function(X_test)
    else:
        y_score = y_pred

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC AUC": roc_auc_score(y_test, y_score),
        "Training + prediction time (s)": elapsed
    })

    trained_models[name] = model

results_df = pd.DataFrame(results).sort_values(
    by="Balanced Accuracy",
    ascending=False
).reset_index(drop=True)

results_df.round(3)


## 5. Visual comparison

Notice that the ranking can change depending on which metric you use.


In [ ]:
plot_df = results_df.set_index("Model")[
    ["Accuracy", "Balanced Accuracy", "F1", "ROC AUC"]
]

ax = plot_df.plot(
    kind="bar",
    figsize=(12, 6),
    ylim=(0.70, 1.01)
)

ax.set_title("Performance of different classifiers on the same test set")
ax.set_ylabel("Score")
ax.set_xlabel("")
ax.legend(loc="lower right")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()


## 6. Confusion matrices

A confusion matrix shows **what kind of errors** each classifier makes.

This can be more informative than one summary number.


In [ ]:
for name, model in trained_models.items():
    ConfusionMatrixDisplay.from_estimator(
        model,
        X_test,
        y_test,
        display_labels=data.target_names,
        cmap="Blues"
    )
    plt.title(name)
    plt.tight_layout()
    plt.show()


## 7. ROC curves

ROC curves compare classifier performance over many possible decision thresholds.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

for name, model in trained_models.items():
    RocCurveDisplay.from_estimator(
        model,
        X_test,
        y_test,
        name=name,
        ax=ax
    )

ax.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
ax.set_title("ROC curves")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


## 8. Which classifier is the best?

Sort the table by different metrics.

You may find that a model that is best by one metric is not necessarily best by another.


In [ ]:
for metric in ["Accuracy", "Balanced Accuracy", "F1", "ROC AUC"]:
    best_row = results_df.loc[results_df[metric].idxmax()]
    print(f"{metric:18s}: {best_row['Model']} ({best_row[metric]:.3f})")


# Discussion questions for students

1. Why do different machine learning algorithms achieve different results on the same dataset?
2. Why do KNN, SVM, Logistic Regression, and MLP use `StandardScaler`, while Decision Tree and Random Forest do not?
3. Is the classifier with the highest **accuracy** automatically the best model?
4. Which error is more important in a medical classification problem: false positive or false negative?
5. Why can a Decision Tree overfit more easily than Logistic Regression?
6. Why can Random Forest perform better than a single Decision Tree?
7. What happens if you change `RANDOM_STATE`?
8. What happens if you change the size of the training set?
9. Would the same classifier still be the best on another dataset?


## 9. Student experiment

Try changing one parameter at a time:

```python
KNeighborsClassifier(n_neighbors=1)
KNeighborsClassifier(n_neighbors=15)

DecisionTreeClassifier(max_depth=2)
DecisionTreeClassifier(max_depth=None)

SVC(C=0.1)
SVC(C=100)

RandomForestClassifier(n_estimators=10)
RandomForestClassifier(n_estimators=500)
```

Then rerun the evaluation and observe how the results change.

### Main conclusion

> Machine learning performance is determined not only by the dataset, but also by the **choice of algorithm, preprocessing, hyperparameters, data split, and evaluation metric**.
